# Yeo-17 Metrics — Schaefer 200-parcel pipeline

Generates `all_subjects_yeo17_metrics.csv` using the correct formula from `utils/metrics_yeo17.py`:
- Same 200-parcel FC matrices as the Yeo-7 pipeline (directly comparable)
- Yeo-17 labels from `utils/matched_labels_exact.csv` (authoritative atlas mapping)
- Correct PC: positive weights only, `tanh` back-transform for seg/int, WMD included

In [1]:
import os
import numpy as np
import pandas as pd
from collections import defaultdict

import sys
PROJECT_ROOT = os.path.abspath('..')
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

from utils.metrics_yeo17 import (
    load_subject,
    compute_all_metrics_yeo17,
    YEO17_ORDER,
    YEO17_ID_TO_NAME,
)

In [2]:
DATA_ROOT  = '../../../Data/'
LOOKUP_CSV = '../utils/matched_labels_exact.csv'

print('Network order (from matched_labels_exact.csv):')
for i, name in enumerate(YEO17_ORDER, 1):
    print(f'  {i:2d}. {name}')

Network order (from matched_labels_exact.csv):
   1. VisCent
   2. VisPeri
   3. SomMotA
   4. SomMotB
   5. DorsAttnA
   6. DorsAttnB
   7. SalVentAttnA
   8. SalVentAttnB
   9. LimbicB
  10. LimbicA
  11. ContA
  12. ContB
  13. ContC
  14. DefaultA
  15. DefaultB
  16. DefaultC
  17. TempPar


## 1. Load subjects

`load_subject` from `metrics_yeo17` loads the **200-parcel** (`_200`) FC matrices
and each subject's own Yeo-7 labels from their `_utils.pkl` file.
Yeo-17 labels are derived later via the lookup table inside `compute_all_metrics_yeo17`.

In [3]:
def discover_subjects(data_root):
    return sorted(
        d for d in os.listdir(data_root)
        if os.path.isdir(os.path.join(data_root, d))
    )

subject_ids = discover_subjects(DATA_ROOT)
print(f'Subjects found: {subject_ids}\n')

all_data = {}
for sid in subject_ids:
    sessions, yeo_labels = load_subject(DATA_ROOT, sid)
    all_data[sid] = {'sessions': sessions, 'yeo_labels': yeo_labels}

print(f'\nLoaded {len(all_data)} subject(s).')

Subjects found: ['002GNV', '003GNV', '004GNV', '005GNV', '008GNV', '012GNV', '013GNV', '016GNV', '020GNV', '021GNV', '023GNV', '025GNV', '026GNV', '034GNV', '037GNV', '041GNV', '043GNV', '048GNV', '049GNV', '054GNV', '056GNV']

  002GNV: 15 session file(s)
  003GNV: 15 session file(s)
  004GNV: 15 session file(s)
  005GNV: 15 session file(s)
  008GNV: 15 session file(s)
  012GNV: 15 session file(s)
  013GNV: 15 session file(s)
  016GNV: 15 session file(s)
  020GNV: 15 session file(s)
  021GNV: 15 session file(s)
  023GNV: 15 session file(s)
  025GNV: 15 session file(s)
  026GNV: 15 session file(s)
  034GNV: 15 session file(s)
  037GNV: 15 session file(s)
  041GNV: 15 session file(s)
  043GNV: 15 session file(s)
  048GNV: 15 session file(s)
  049GNV: 15 session file(s)
  054GNV: 15 session file(s)
  056GNV: 15 session file(s)

Loaded 21 subject(s).


In [4]:
# Remove the extra Run8 present only in 012GNV V03
for sid, subj_data in all_data.items():
    sessions = subj_data['sessions']
    if 'V03' in sessions and 'Run8' in sessions['V03']:
        del sessions['V03']['Run8']
        print(f'Removed V03/Run8 from {sid}')

Removed V03/Run8 from 012GNV


In [5]:
# Verify run counts per session
counts = defaultdict(int)
for sid, subj_data in all_data.items():
    for sess, runs in subj_data['sessions'].items():
        for run in runs:
            counts[(sess, run)] += 1

rows = [{'session': sess, 'run': run, 'n_subjects': n}
        for (sess, run), n in counts.items()]
table = pd.DataFrame(rows).pivot(index='session', columns='run', values='n_subjects')
print(table)

run      Run1  Run2  Run3  Run4  Run5  Run6  Run7
session                                          
V01        21    21    21    21    21    21     8
V02        21    21    21    21    21    21     7
V03        21    21    21    21    21    21     7
V04        21    21    21    21    21    20     6
V05        21    20    21    21    21    21     7
V06        21    21    21    21    21    21     6
V07        21    21    21    19    21    20     8
V08        21    21    21    21    21    21     8
V09        21    21    21    21    21    20     6
V10        21    21    21    21    21    21     7
V11        21    21    21    20    21    21     5
V12        21    21    21    21    21    21     8
V13        21    21    21    21    21    21     7
V14        21    21    21    21    21    21     9
V15        21    21    21    21    21    21     7


## 2. Compute metrics

`compute_all_metrics_yeo17` does the following for every run:
1. Converts each subject's Yeo-7 labels → Yeo-17 labels via `matched_labels_exact.csv`
2. Restricts to cortical parcels (Yeo-17 label 1–17)
3. Fisher z-transforms the FC matrix
4. Computes segregation and integration (mean in z-space, back-transformed with `tanh`)
5. Computes PC with **positive weights only** (clipped to [0, 1])
6. Computes within-module degree z-score (WMD)

In [6]:
compute_all_metrics_yeo17(all_data, LOOKUP_CSV)
print('Metrics computed for all subjects.')

Metrics computed for all subjects.


In [7]:
# Sanity check on one run
sid  = next(iter(all_data))
sess = next(iter(all_data[sid]['sessions']))
run  = next(iter(all_data[sid]['sessions'][sess]))
rd   = all_data[sid]['sessions'][sess][run]

print(f'Subject: {sid}  Session: {sess}  Run: {run}')
print(f'  condition          : {rd["condition"]}')
print(f'  pc_parcel17  shape : {rd["pc_parcel17"].shape}  '
      f'min={rd["pc_parcel17"].min():.3f}  max={rd["pc_parcel17"].max():.3f}')
print(f'  wmd_parcel17 shape : {rd["wmd_parcel17"].shape}')
print(f'  segregation17      : {np.round(rd["segregation17"], 3)}')
print(f'  integration17      : {np.round(rd["integration17"], 3)}')

Subject: 002GNV  Session: V01  Run: Run1
  condition          : Feedback
  pc_parcel17  shape : (200,)  min=0.896  max=0.932
  wmd_parcel17 shape : (200,)
  segregation17      : [0.596 0.738 0.618 0.607 0.672 0.71  0.631 0.611 0.212 0.443 0.519 0.529
 0.602 0.379 0.509 0.669 0.688]
  integration17      : [0.378 0.406 0.417 0.444 0.457 0.455 0.46  0.427 0.241 0.336 0.439 0.409
 0.441 0.291 0.354 0.336 0.472]


## 3. Export

One row per cortical parcel per run. Network-level metrics (segregation, integration,
normalized_segregation) are repeated on every parcel row for convenience — same
structure as `all_subjects_yeo7_metrics.csv`.

In [8]:
export_rows = []

for sid, subj in all_data.items():
    yeo17 = subj['yeo17_labels']                          # set by compute_all_metrics_yeo17
    cortical_mask    = (yeo17 >= 1) & (yeo17 <= 17)
    yeo_cx           = yeo17[cortical_mask].astype(int)   # (n_cortical,) labels 1-17
    cortical_indices = np.where(cortical_mask)[0]          # original parcel indices

    for sess_id, sess_runs in subj['sessions'].items():
        for run_key in sorted(sess_runs, key=lambda x: int(x.replace('Run', ''))):
            rd      = sess_runs[run_key]
            pc      = rd['pc_parcel17']
            pc_neg  = rd['pc_parcel_neg17']
            wmd     = rd['wmd_parcel17']

            for i, (orig_idx, yeo_lbl) in enumerate(zip(cortical_indices, yeo_cx)):
                net_name = YEO17_ID_TO_NAME[yeo_lbl]
                net_idx  = yeo_lbl - 1
                export_rows.append({
                    'subject':                sid,
                    'session':                sess_id,
                    'run':                    run_key,
                    'condition':              rd['condition'],
                    'parcel':                 int(orig_idx),
                    'yeo_label':              int(yeo_lbl),
                    'pc_parcel':              float(pc[i]),
                    'pc_parcel_neg':          float(pc_neg[i]),
                    'wmd_parcel':             float(wmd[i]),
                    'network':                net_name,
                    'segregation':            float(rd['segregation17'][net_idx]),
                    'integration':            float(rd['integration17'][net_idx]),
                    'normalized_segregation': float(rd['normalized_segregation17'][net_idx]),
                })

df_export = pd.DataFrame(export_rows)
out_path  = os.path.join(DATA_ROOT, 'all_subjects_yeo17_metrics.csv')
df_export.to_csv(out_path, index=False)

print(f'Saved {len(df_export):,} rows to {out_path}')
print(f'Columns: {df_export.columns.tolist()}')
df_export.head(5)

Saved 397,800 rows to ../../../Data/all_subjects_yeo17_metrics.csv
Columns: ['subject', 'session', 'run', 'condition', 'parcel', 'yeo_label', 'pc_parcel', 'pc_parcel_neg', 'wmd_parcel', 'network', 'segregation', 'integration', 'normalized_segregation']


,subject,session,run,condition,parcel,yeo_label,pc_parcel,pc_parcel_neg,wmd_parcel,network,segregation,integration,normalized_segregation
0,002GNV,V01,Run1,Feedback,0,2,0.929990,NaN,-0.758881,VisPeri,0.738021,0.405924,0.290308
1,002GNV,V01,Run1,Feedback,1,1,0.930220,0.473051,2.230399,VisCent,0.595731,0.377746,0.223924
2,002GNV,V01,Run1,Feedback,2,5,0.928906,0.607157,-0.187392,DorsAttnA,0.671652,0.456957,0.190230
3,002GNV,V01,Run1,Feedback,3,2,0.927242,0.000000,0.710269,VisPeri,0.738021,0.405924,0.290308
4,002GNV,V01,Run1,Feedback,4,1,0.932369,NaN,-1.083012,VisCent,0.595731,0.377746,0.223924


In [9]:
# Verify PC values are bounded
col = 'pc_parcel'
print('=== PC sanity check ===')
print(f'  min  = {df_export[col].min():.6f}')
print(f'  max  = {df_export[col].max():.6f}')
print(f'  NaN  = {df_export[col].isna().sum()}')
print(f'  < 0  = {(df_export[col] < 0).sum()}')
print(f'  > 1  = {(df_export[col] > 1).sum()}')
print()
print('=== Parcels per network ===')
print(df_export[df_export['session'] == 'V01'][df_export['run'] == 'Run1']
      [df_export['subject'] == df_export['subject'].iloc[0]]
      .groupby('network')['parcel'].count()
      .reindex(YEO17_ORDER))

=== PC sanity check ===
  min  = 0.144989
  max  = 0.937128
  NaN  = 0
  < 0  = 0
  > 1  = 0

=== Parcels per network ===
network
VisCent         12
VisPeri         12
SomMotA         19
SomMotB         15
DorsAttnA       12
DorsAttnB       10
SalVentAttnA    16
SalVentAttnB    10
LimbicB          6
LimbicA          8
ContA           16
ContB           15
ContC            6
DefaultA        14
DefaultB        17
DefaultC         6
TempPar          6
Name: parcel, dtype: int64


C:\Users\tesni\AppData\Local\Temp\ipykernel_20392\4115760089.py:11: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  print(df_export[df_export['session'] == 'V01'][df_export['run'] == 'Run1']
C:\Users\tesni\AppData\Local\Temp\ipykernel_20392\4115760089.py:11: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  print(df_export[df_export['session'] == 'V01'][df_export['run'] == 'Run1']
